# 14a — LPT spherical density at 32³: MUSE (pixel + harmonic)

MUSE marginal inference of `(Ω_c, σ_8)` over the 32³ white initial-condition field, on the LPT → spherical galaxy-overdensity forward model, in both the pixel and the harmonic (ℓ-tapered scale cut) likelihood. Hard gates: MUSE-pixel recovers the truth within its own uncertainty, and MUSE-harmonic agrees with MUSE-pixel.

Same config + mock (seed 0) as `14b-LPTDensityNUTS` and `14c-LPTDensityMCLMC` — the three posteriors are directly comparable. Run headless with `uv run --no-sync papermill 14a-LPTDensityMUSE.ipynb 14a-LPTDensityMUSE.ipynb --cwd .`; x64 everywhere; the inner MAP is L-BFGS with a shared real-data-MAP warm start for all simulations.

In [ ]:
%load_ext autoreload
%autoreload 2
import os

os.environ["JAX_ENABLE_X64"] = "True"  # float32 => NaN/chaotic IC gradients
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")  # coexist with other GPU processes
# At 32^3 the pipeline is a chain of small sequential ops: a many-core CPU can beat a small GPU
# (measured: 0.14 s/grad on a 20-core CPU vs 0.51 s on an RTX 4060). Set JAX_PLATFORMS=cpu to force CPU.
os.environ.setdefault("JAX_PLATFORMS", "cuda,cpu")

import dataclasses
import time

import jax
import jax.numpy as jnp
import jax_cosmo as jc
import matplotlib.pyplot as plt
import numpy as np
import jax_fli as jfli
from jax.scipy.special import ndtr
from numpyro.handlers import condition, seed, trace

jax.config.update("jax_enable_x64", True)
from jax_fli.infer.muse import muse_inference, muse_problem_from_model

MESH = 32  # laptop/cluster-node size; bump for production or use the distributed 15-lensing-muse-inference.py
NSIDE = MESH
print(f"jax {jax.__version__}  backend {jax.default_backend()}  x64 {jax.config.jax_enable_x64}  MESH={MESH}")

## 1. Model configuration and mock observation

LPT-only (`sim_mode="lpt"`) → spherical galaxy overdensity (`lensing_output="density"`), two tomographic source planes at z = 0.10 and 0.17 inside the z ≤ 0.2 lightcone, centered observer (whole sky). The mock is a forward draw of the pixel model at a random prior point (seed 0 — identical across the 14a/14b/14c notebooks, so their posteriors are directly comparable); we condition on it and warm-start at the truth.

In [ ]:
cosmo = jc.Planck18()
box = tuple(float(x) for x in jfli.utils.compute_box_size_from_redshift(cosmo, 0.2, (0.5, 0.5, 0.5)))
priors = {
    "Omega_c": jfli.infer.PreconditionnedUniform(0.1, 0.5),
    "sigma8": jfli.infer.PreconditionnedUniform(0.6, 1.0),
}

config = jfli.ppl.Configurations(
    mesh_size=(MESH, MESH, MESH),
    box_size=box,
    halo_size=(0, 0),
    field_sharding=None,
    sim_mode="lpt",
    nbody_solver="BullFrog",
    t0=0.001,
    t1=1.0,
    lpt_order=1,
    number_of_shells=5,
    nb_steps=5,
    paint_order="cic",
    gradient_order=4,
    laplace_fd=True,
    shell_spacing="a",
    time_stepping="D",
    min_width=1.0,
    lensing_output="density",
    map2alm_method="jax",
    likelihood_space="pixel",
    min_redshift=0.001,
    max_redshift=0.2,
    n_integrate=8,
    nside=NSIDE,
    geometry="spherical",
    scheme="rbf_neighbor",
    observer_position=(0.5, 0.5, 0.5),
    paint_nside=NSIDE,
    kernel_width_pixels=0.8,
    fiducial_cosmology=jc.Planck18,
    nz_shear=[0.10, 0.17],
    priors=priors,
    sigma_e=0.3,
    adjoint="checkpointed",
    checkpoints=2,
)

pixel_model = jfli.ppl.full_field_probmodel(config)
tr = trace(seed(pixel_model, 0)).get_trace()
x_obs = jnp.stack([v["value"] for k, v in tr.items() if "observable" in k and k != "observable_meta_data"], axis=0)
x_obs_meta_data = tr["observable_meta_data"]["value"]
theta_truth = jnp.array([float(tr["Omega_c_base"]["value"]), float(tr["sigma8_base"]["value"])])
Oc_true, s8_true = float(tr["Omega_c"]["value"]), float(tr["sigma8"]["value"])
truth = {"Omega_c": Oc_true, "sigma8": s8_true}
print(f"truth: Omega_c={Oc_true:.4f}  sigma8={s8_true:.4f}   observable {x_obs.shape}")

# Harmonic variant of the same config: NEVER mutate the shared dataclass in place — use dataclasses.replace.
ell_max, taper = min(2 * NSIDE - 1, 3 * NSIDE // 2), 4
config_h = dataclasses.replace(config, likelihood_space="harmonic", ell_max=int(ell_max), ell_taper_width=taper)
harmonic_model = jfli.ppl.full_field_probmodel(config_h, observed_maps=x_obs)

In [ ]:
jfli.SphericalDensity.FromDensityMetadata(array=x_obs, field=x_obs_meta_data).show()

## 2. Sanity gate: jitted log-density + gradient at the truth

Expect **large** cosmology-base gradients (O(10²–10³) at 32³): at the true parameters the score is a zero-mean random variable with std = √Fisher, and a field-level likelihood has a big Fisher information for 2 parameters. This is expected statistics, not a normalization bug (no noise variance depends on the sampled cosmology) — see `docs/WORK_IN_PROGRESS/20-likelihood-audit.md`. Time gradients **under `jax.jit`**: an eager `jax.grad` call is dispatch-bound and ~50× slower at this size (the historical "8 s per gradient" was that artifact).

In [ ]:
data = {f"observable_{i}": x_obs[i] for i in range(x_obs.shape[0])}
cond_model = condition(pixel_model, data=data)

from numpyro.infer.util import initialize_model

init, potential_fn, postprocess_fn, model_trace = initialize_model(jax.random.key(0), cond_model)
logdensity_fn = lambda position: -potential_fn(position)
value_and_grad_fn = jax.jit(jax.value_and_grad(logdensity_fn))

truth_position = {
    "Omega_c_base": theta_truth[0],
    "sigma8_base": theta_truth[1],
    "initial_conditions": tr["initial_conditions"]["value"].array,
}
t0 = time.time()
val, grad = jax.block_until_ready(value_and_grad_fn(truth_position))
print(f"compile+first-run: {time.time() - t0:.1f} s")
t0 = time.time()
for _ in range(3):
    val, grad = jax.block_until_ready(value_and_grad_fn(truth_position))
print(f"jitted gradient: {(time.time() - t0) / 3 * 1e3:.0f} ms")
assert np.isfinite(float(val)) and all(bool(np.all(np.isfinite(np.asarray(g)))) for g in jax.tree.leaves(grad))
print(f"logp={float(val):.1f}  dOc_base={float(grad['Omega_c_base']):.3e}  ds8_base={float(grad['sigma8_base']):.3e}")

## 3. MUSE: pixel, then harmonic

`muse_problem_from_model(config)` builds the problem straight from the NumPyro model (θ = the white cosmology bases, latent z = the white IC field). The posterior is reported by drawing `N(θ̂, Σ)` in white space and pushing the samples through the `PreconditionnedUniform` bijector `Ω = lo + (hi−lo)·Φ(base)` (transform samples, not the covariance). The score is a cosmology gradient at the inner MAP (envelope theorem) — its O(10²–10³) magnitude is handled by MUSE natively (Newton steps use the score covariance over simulations, which is self-scaling).

In [ ]:
prob = muse_problem_from_model(config)
prob_h = muse_problem_from_model(config_h)

MUSE_KW = dict(n_sims=20, maxsteps=10, n_sims_cov=30, n_sims_H=5, map_maxiter=1000, map_gtol=1e-3, progress=True)

t0 = time.time()
res_pix = muse_inference(
    prob, x_obs, path="output/nb14a/muse_pix", rng_key=jax.random.PRNGKey(1), theta0=theta_truth, **MUSE_KW
)
print(f"MUSE pixel: {time.time() - t0:.0f}s, {res_pix.n_steps} steps, gnorm={res_pix.map_gnorm:.1e}")

t0 = time.time()
res_h = muse_inference(
    prob_h, x_obs, path="output/nb14a/muse_harm", rng_key=jax.random.PRNGKey(2), theta0=theta_truth, **MUSE_KW
)
print(f"MUSE harmonic: {time.time() - t0:.0f}s, {res_h.n_steps} steps, gnorm={res_h.map_gnorm:.1e}")

In [ ]:
def bases_to_cosmo(base_draws):
    return {n: np.asarray(p.low + (p.high - p.low) * ndtr(base_draws[:, i])) for i, (n, p) in enumerate(priors.items())}


def muse_extract(name, res, seed_):
    draws = jax.random.multivariate_normal(jax.random.PRNGKey(seed_), res.theta, res.Sigma, shape=(4000,))
    cosmo_draws = {k: v[None, :] for k, v in bases_to_cosmo(draws).items()}
    return jfli.io.CatalogExtract(name=name, cosmo=cosmo_draws, truth_cosmo=truth)


extracts = {"MUSE-pixel": muse_extract("MUSE-pixel", res_pix, 11), "MUSE-harmonic": muse_extract("MUSE-harmonic", res_h, 12)}

# space-consistency: the pixel and harmonic centers must agree within the larger of the two spreads
for i, k in enumerate(("Omega_c", "sigma8")):
    pi = extracts["MUSE-pixel"].cosmo[k].ravel()
    ha = extracts["MUSE-harmonic"].cosmo[k].ravel()
    assert abs(pi.mean() - ha.mean()) < 2.0 * max(pi.std(), ha.std()), f"pixel vs harmonic {k} centers disagree"
print("PASS: MUSE pixel ~ MUSE harmonic")

In [ ]:
S = {name: {k: (ex.cosmo[k].ravel().mean(), ex.cosmo[k].ravel().std()) for k in ("Omega_c", "sigma8")} for name, ex in extracts.items()}
for name, s in S.items():
    print(
        f"{name:16s}  Omega_c={s['Omega_c'][0]:.3f}+/-{s['Omega_c'][1]:.3f}   sigma8={s['sigma8'][0]:.3f}+/-{s['sigma8'][1]:.3f}"
    )
print(f"{'truth':16s}  Omega_c={truth['Omega_c']:.3f}           sigma8={truth['sigma8']:.3f}")

for name, s in S.items():
    for k in ("Omega_c", "sigma8"):
        m, sd = s[k]
        assert abs(m - truth[k]) < 3.0 * max(sd, 1e-6), f"{name} {k}={m:.3f} off truth {truth[k]:.3f} (sigma {sd:.3f})"
print("PASS: every run recovers the true cosmology within 3 sigma of its own spread")

jfli.infer.plot_posterior(list(extracts.values()), labels={"Omega_c": r"\Omega_c", "sigma8": r"\sigma_8"})
plt.show()

## Methods note

MUSE marginalizes the IC field by scoring at its MAP and debiasing with simulations (`n_sims` per Newton step); `Σ = (Hᵀ J⁻¹ H + Λ_prior)⁻¹` from `n_sims_cov` fresh scores (J) and `n_sims_H` central-difference response sims (H). The inner MAP must converge (`map_gnorm ≲ 10·map_gtol`) for the envelope-theorem score to be exact — if it does not, raise `map_maxiter` (1000 → 3000) before loosening `map_gtol`. MUSE recovering the truth also validates the density shot-noise model `N_ℓ = 1/n̄` flagged unvalidated in `number_counts`. The harmonic likelihood is a classic observed site (`harmonic_obs`), exactly equivalent to the historical packed-residual factor.